# 1280. Students and Examinations

## Problem
Students(student_id,student_name), Subjects(subject_name), Examinations(student_id,subject_name): report all student-subject combinations and their exam counts (including zero). Return student_id,student_name,subject_name,attended_exams sorted by student_id,subject_name.

[Doocs problem reference](https://github.com/doocs/leetcode/blob/main/solution/1200-1299/1280.Students%20and%20Examinations/README_EN.md) · [LeetCode](https://leetcode.com/problems/students-and-examinations/)

[Community solution index (Silversaya, 2023)](https://leetcode.cn/discuss/post/3573106/zhe-shi-pandas30tian-ji-hua-de-ti-jie-qu-v0wv/) · [LeetCode discussions](https://leetcode.com/problems/students-and-examinations/solutions/) · [Individual community solution](https://leetcode.cn/problems/students-and-examinations/solutions/2379851/1280-xue-sheng-men-can-jia-ge-ke-ce-shi-sg5gz/)

Community links are for further reading; the implementations below are independently written.

## Practice first
Run the data cell and try the blank cell before reading solutions.

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal

students = pd.DataFrame([[1,"A"],[2,"B"]], columns=["student_id","student_name"])
subjects = pd.DataFrame([["Math"],["Art"]], columns=["subject_name"])
examinations = pd.DataFrame([[1,"Math"],[1,"Math"],[2,"Art"]], columns=["student_id","subject_name"])
expected = pd.DataFrame([[1,"A","Art",0],[1,"A","Math",2],[2,"B","Art",1],[2,"B","Math",0]], columns=["student_id","student_name","subject_name","attended_exams"])
display(students)
display(subjects)
display(examinations)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions

### Solution 1 — Cross merge and counts
先建立完整母集合，才不會漏零次考試。

In [ ]:
def solution_1(students, subjects, examinations):
    keys = ['student_id','subject_name']
    g = examinations.groupby(keys).size().reset_index(name='attended_exams')
    out = students.merge(subjects,how='cross').merge(g,on=keys,how='left')
    out['attended_exams'] = out.attended_exams.fillna(0).astype(int)
    return out[['student_id','student_name','subject_name','attended_exams']].sort_values(keys)

result_1 = solution_1(students.copy(), subjects.copy(), examinations.copy())
display(result_1)

### Solution 2 — MultiIndex reindex
MultiIndex 組合完整鍵，再補零。

In [ ]:
def solution_2(students, subjects, examinations):
    keys = ['student_id','subject_name']
    idx = pd.MultiIndex.from_product([students.student_id,subjects.subject_name],names=keys)
    g = examinations.groupby(keys).size().reindex(idx,fill_value=0).reset_index(name='attended_exams')
    return g.merge(students,on='student_id')[['student_id','student_name','subject_name','attended_exams']].sort_values(keys)

result_2 = solution_2(students.copy(), subjects.copy(), examinations.copy())
display(result_2)

### Solution 3 — Crosstab and stack
crosstab 生成矩陣，reindex 補齊，再 stack 還原長表。

In [ ]:
def solution_3(students, subjects, examinations):
    keys = ['student_id','subject_name']
    p = pd.crosstab(examinations.student_id,examinations.subject_name).reindex(index=students.student_id,columns=subjects.subject_name,fill_value=0).rename_axis(index='student_id',columns='subject_name')
    g = p.stack().reset_index(name='attended_exams')
    return g.merge(students,on='student_id')[['student_id','student_name','subject_name','attended_exams']].sort_values(keys)

result_3 = solution_3(students.copy(), subjects.copy(), examinations.copy())
display(result_3)

## Check all solutions
Rows are normalized only for the example comparison; preserve required ordering in the solutions themselves.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Compare row grain, missing matches, duplicate keys and zero-count categories.